In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [2]:
df = pd.read_csv("C:\\Users\\Lenovo\\Downloads\\ecommerce_customer_churn.csv")

X = df.drop(columns=['CustomerID', 'Churn'])
y = df['Churn']

In [3]:
num_col = ['Age', 'TenureMonths', 'MonthlyCharges', 'TotalCharges']
cat_col = ['ContractType', 'PaperlessBilling', 'PaymentMethod']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [8]:
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

In [9]:
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_col),
    ('cat', cat_transformer, cat_col)
])

In [18]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42))
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=50, random_state=42))
])

gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', GradientBoostingClassifier(random_state=42))
])

In [19]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("--- Cross-Validation Model Comparison ---")
for name, pipeline in [("Logistic Regression", lr_pipeline),
                       ("Random Forest", rf_pipeline),
                       ("Gradient Boosting", gb_pipeline)]:
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
    print(f"{name} CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

--- Cross-Validation Model Comparison ---
Logistic Regression CV Accuracy: 0.6700 (+/- 0.0388)
Random Forest CV Accuracy: 0.6412 (+/- 0.0368)
Gradient Boosting CV Accuracy: 0.6550 (+/- 0.0307)


In [20]:
param_grid = {
    'clf__C': [0.01, 0.1, 1.0, 10.0],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs']
}

grid_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\n--- Hyperparameter Tuning ---")
print("Best Parameters:", grid_search.best_params_)
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")


--- Hyperparameter Tuning ---
Best Parameters: {'clf__C': 10.0, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}
Best CV ROC-AUC: 0.7236


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [13]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("\n--- Test Set Evaluation ---")
print(classification_report(y_test, y_pred))
print(f"Test Set ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))


--- Test Set Evaluation ---
              precision    recall  f1-score   support

           0       0.74      0.76      0.75       130
           1       0.54      0.51      0.53        70

    accuracy                           0.68       200
   macro avg       0.64      0.64      0.64       200
weighted avg       0.67      0.68      0.67       200

Test Set ROC-AUC Score: 0.7155

--- Confusion Matrix ---
[[99 31]
 [34 36]]


In [14]:
cat_encoder = best_model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder']
encoded_cat_cols = cat_encoder.get_feature_names_out(cat_col)
all_feature_names = num_col + list(encoded_cat_cols)

In [15]:
coefs = best_model.named_steps['clf'].coef_[0]

feature_importance = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': coefs,
    'Abs_Coef': np.abs(coefs)
}).sort_values(by='Abs_Coef', ascending=False)

print("\n--- Top Features Driving Customer Churn ---")
print(feature_importance[['Feature', 'Coefficient']].to_string(index=False))

# 9. Save Model Pipeline
model_filename = 'customer_churn_pipeline.pkl'
joblib.dump(best_model, model_filename)
print(f"\nTrained pipeline successfully saved as '{model_filename}'.")


--- Top Features Driving Customer Churn ---
                       Feature  Coefficient
         ContractType_Two Year    -1.564686
         ContractType_One Year    -1.526404
                  TenureMonths    -0.590672
    PaymentMethod_Mailed Check     0.355286
PaymentMethod_Electronic Check     0.326155
                  TotalCharges     0.313596
     PaymentMethod_Credit Card     0.190143
                MonthlyCharges     0.120888
                           Age     0.075426
          PaperlessBilling_Yes     0.029552

Trained pipeline successfully saved as 'customer_churn_pipeline.pkl'.
